In [ ]:
# ==============================================================================
# PARALLEL FACT: SERVICE
# ==============================================================================
from helpers import IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger, safe_count, generate_batch_id
import pandas as pd
from helpers.silver_transforms import transform_service_fact

logger = setup_logger("parallel_fact_service")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

config = TableConfig(
    table_name="service",
    business_key="service_id",
    surrogate_key="service_key",
    watermark_column="service_date",
    scd_type=1,
    gold_table_name="fact_service",
    silver_transform=transform_service_fact,
)

bronze_batch_id = get_latest_batch_id(spark, "service")
if not bronze_batch_id:
    raise ValueError("No bronze batch_id found for service; run bronze load first.")
logger.info(f"Using bronze batch_id for service: {bronze_batch_id}")

print("Row Counts (Before):")
print(f"fact_service: {safe_count(spark, 'fact_service')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))

results = pipeline.load_tables([config], force_full=False, bronze_batch_id=bronze_batch_id)
display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"fact_service: {safe_count(spark, 'fact_service')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
